# Social choice assignment project CAI

We will analyse the dataset 00073-00000002.cat regarding the 2017 categorization of French electoral candidates. We will do this by applying various social choice voting rules and construct & evaluate their social choice functions. 

The data is categorical, with a ranking within the categories (1, 0, -1) possibly representing approve, neutral, disapprove. Many candidates may be ranked in any category. So then, we must count their category as their preference ranking and cannot assume any ranking within a preference category. 

## Loading the Data into Python

In [1]:
import re

def load_cat(filepath):
    metadata = {}
    ballots = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Metadata
            if line.startswith('#'):
                if ':' in line:
                    key, value = line[1:].split(':', 1)
                    metadata[key.strip()] = value.strip()
                continue

            # Skip lines that don't look like ballots
            if ':' not in line:
                continue

            count_str, prefs_str = line.split(':', 1)
            count = int(count_str.strip())

            groups = re.findall(r'\{([^}]*)\}', prefs_str)
            ranking = [
                {int(x.strip()) for x in group.split(',') if x.strip()}
                for group in groups
            ]

            ballots.append({
                "count": count,
                "ranking": ranking
            })


    return metadata, ballots

In [2]:
metadata, ballots = load_cat("00073-00000002.cat")

total_voters = metadata.get("NUMBER VOTERS", "Unknown")
print(f"Total voters: {total_voters}")

alternatives = metadata.get("NUMBER ALTERNATIVES", "Unknown")
print(f"Number of alternatives: {alternatives}")

print(ballots[0])

Total voters: 13197
Number of alternatives: 11
{'count': 238, 'ranking': [{10, 6}, {1, 11, 9}, {2, 3, 4, 5, 7, 8}]}


The count defines our score value in the scoring vector, while the ranking (preferences) have been stores in the ranking attribute. 

# Plurality Rule

We choose the best candidate by the number of times they are number 1 in the preferences. Hence this is the scoring rule with the scoring vector s = (1, 0,0,.., 0) for the U (universe of alternatives in the preferences). However, we will only look at the best category, so hence we can look at how many times each alternative (candidate) is in the category 1. To implement this scoring rule we apply for each preference casted, this score vector. 

In [3]:
def plurality_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        if ranking:
            top_group = ranking[0]
            for candidate in top_group:
                scores[candidate] = scores.get(candidate, 0) + count
    max_score = max(scores.values())
    print(scores)
    winners = [candidate for candidate, score in scores.items() if score == max_score]
    return winners

In [4]:
plurality_winner_idx = plurality_rule_SCF(ballots)

for idx in plurality_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

{10: 8621, 6: 8208, 1: 3308, 11: 5449, 9: 3985, 2: 1253, 3: 1327, 4: 1666, 5: 1437, 7: 2160, 8: 854}
Jean-Luc Mélenchon


# Anti - Plurality Rule (Veto Rule)

For the anti-plurality rule we reward alternatives who are not last, with a scoring vector of (1,1,...,0). We can thus find out how many times each alternative falls in the final category and then award the ones who do this least with the winner title. 


In [5]:
# anti_plurality_score_vector = [1 if i < alternatives else 0 for i in range(alternatives)]
def anti_plurality_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        if ranking:
            bottom_group = ranking[-1]
            for candidate in bottom_group:
                scores[candidate] = scores.get(candidate, 0) + count
    min_score = min(scores.values())
    print(scores)
    winners = [candidate for candidate, score in scores.items() if score == min_score]
    return winners

In [6]:
anti_plurality_winner_idx = anti_plurality_rule_SCF(ballots)
for idx in anti_plurality_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

{2: 8643, 3: 7525, 4: 8446, 5: 9711, 7: 5894, 8: 10800, 9: 4586, 1: 4508, 11: 3484, 10: 1711, 6: 1682}
Benoît Hamon


# Borda's Rule

With Borda's rule we award the alternatives by their rank. Hence the scoring vector is s = (n-1, n-2, ..., 0). We have 3 categorical ranks, so we have (2, 1, 0) and we will award each alternative falling into a rank with the scoring[rank] value

In [7]:
# borda_score_vector = [2, 1, 0]
def borda_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        num_groups = len(ranking)
        for i, group in enumerate(ranking):
            points = num_groups - i - 1
            for candidate in group:
                scores[candidate] = scores.get(candidate, 0) + points * count
    max_score = max(scores.values())
    print(scores)
    winners = [candidate for candidate, score in scores.items() if score == max_score]
    return winners

In [8]:
borda_winner_idx = borda_rule_SCF(ballots)
for idx in borda_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

{10: 15676, 6: 15216, 1: 8574, 11: 11431, 9: 8799, 2: 3258, 3: 4327, 4: 3807, 5: 2464, 7: 6397, 8: 1204}
Jean-Luc Mélenchon


So far, the 3 scoring rules do not finding the condorcet winner in some preference profiles. Hence we will consider a condorcet extension too. 

# Copeland's Rule

Copeland's rule awards an alternative for a pairwaise result, win, lose or tie. Here we will consider the ties within a category to count as well with a value between 0 and 1. 

In [9]:
tie_value = 0.5

def copeland_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        for i, group in enumerate(ranking):
            for candidate in group:
                scores[candidate] = scores.get(candidate, 0) + sum(len(ranking[j]) for j in range(i+1, len(ranking))) * count # how many wins
                scores[candidate] = scores.get(candidate, 0) - sum(len(ranking[j]) for j in range(i)) * count # how many losses
                scores[candidate] = scores.get(candidate, 0) + tie_value * count*(len(group)-1) # how many ties
    max_score = max(scores.values())
    print(scores)
    winners = [candidate for candidate, score in scores.items() if score == max_score]
    return winners

In [10]:
copeland_winner_idx = copeland_rule_SCF(ballots)
for idx in copeland_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

{10: 73204.5, 6: 70014.5, 1: 34027.0, 11: 51071.5, 9: 30065.5, 2: -2030.5, 3: 6545.0, 4: 1220.5, 5: -12485.0, 7: 20754.5, 8: -20377.5}
Jean-Luc Mélenchon


In [11]:
# Lets change the value of tie_value and see how it affects the winner
# Lets plot this as well for different values of tie_value

tie_values = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
copeland_winners = {}
for tie in tie_values:
    tie_value = tie
    copeland_winner_idx = copeland_rule_SCF(ballots)
    copeland_winners[tie] = [metadata[f"ALTERNATIVE NAME {idx}"] for idx in copeland_winner_idx]
print(copeland_winners)

{10: 57880, 6: 54956, 1: 11018, 11: 29613, 9: 11611, 2: -29588, 3: -20698, 4: -25706, 5: -37294, 7: -4577, 8: -47215}
{10: 60944.89999999858, 6: 57967.69999999877, 1: 15619.799999999888, 11: 33904.699999999924, 9: 15301.900000000209, 2: -24076.500000000087, 3: -15249.400000000116, 4: -20320.700000000154, 5: -32332.200000000244, 7: 489.2999999999899, 8: -41847.49999999978}
{10: 64009.799999999006, 6: 60979.39999999923, 1: 20221.599999999926, 11: 38196.39999999977, 9: 18992.800000000272, 2: -18565.00000000019, 3: -9800.799999999954, 4: -14935.399999999934, 5: -27370.399999999976, 7: 5555.6, 8: -36480.000000000175}
{10: 67074.70000000115, 6: 63991.100000000784, 1: 24823.400000000078, 11: 42488.10000000028, 9: 22683.699999999608, 2: -13053.500000000045, 3: -4352.200000000058, 4: -9550.100000000077, 5: -22408.600000000028, 7: 10621.89999999999, 8: -31112.499999999905}
{10: 70139.60000000025, 6: 67002.80000000085, 1: 29425.200000000135, 11: 46779.80000000021, 9: 26374.600000000348, 2: -7541.

# Single transferable vote (STV)


We look at the alternative that is winning (First place) the least and remove. We continue this process till we have a winner. Everytime we remove an agent, we must only drop that agent from all preferences. This way, other agents, in 2nd or lower places may end up surviving till the end. 

In [16]:
def tally_first(ballots):
    # Let us tally the 1st place (1st category) wins for each candidate
    first_place_counts = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        if ranking:
            top_group = ranking[0]
            for candidate in top_group:
                first_place_counts[candidate] = first_place_counts.get(candidate, 0) + count
    return first_place_counts

# we find the candidate(s) with the least 1st place votes and eliminate them, then we repeat the process until one candidate remains
def single_transferable_vote_SCF(ballots):
    remaining_candidates = set()
    for ballot in ballots:
        for group in ballot["ranking"]:
            remaining_candidates.update(group)

    while len(remaining_candidates) > 1:
        first_place_counts = tally_first(ballots)
        min_count = min(first_place_counts[candidate] for candidate in remaining_candidates)
        candidates_to_eliminate = [candidate for candidate in remaining_candidates if first_place_counts[candidate] == min_count]
        print(f"Eliminating candidates: {[metadata[f'ALTERNATIVE NAME {idx}'] for idx in candidates_to_eliminate]} with {min_count} first-place votes")
        # Eliminate candidates
        remaining_candidates -= set(candidates_to_eliminate)

        # Remove eliminated candidates from ballots
        new_ballots = []
        for ballot in ballots:
            count = ballot["count"]
            ranking = ballot["ranking"]
            new_ranking = []
            for group in ranking:
                new_group = {candidate for candidate in group if candidate in remaining_candidates}
                if new_group:
                    new_ranking.append(new_group)
            if new_ranking:
                new_ballots.append({"count": count, "ranking": new_ranking})
        ballots = new_ballots

    return list(remaining_candidates)   

In [17]:
stv_winner_idx = single_transferable_vote_SCF(ballots)
for idx in stv_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

Eliminating candidates: ['Marine Le Pen'] with 854 first-place votes
Eliminating candidates: ['François Asselineau'] with 1447 first-place votes
Eliminating candidates: ['Jacques Cheminade'] with 1547 first-place votes
Eliminating candidates: ['François Fillon'] with 1675 first-place votes
Eliminating candidates: ['Nicolas Dupont-Aignan'] with 1938 first-place votes
Eliminating candidates: ['Jean Lassalle'] with 2608 first-place votes
Eliminating candidates: ['Nathalie Arthaud'] with 3794 first-place votes
Eliminating candidates: ['Emmanuel Macron'] with 4610 first-place votes
Eliminating candidates: ['Philippe Poutou'] with 6367 first-place votes
Eliminating candidates: ['Benoît Hamon'] with 9564 first-place votes
Jean-Luc Mélenchon


# Is there a Condorcet Winner

In [14]:
def condorcet_winner_SCF(ballots):
    candidates = set()
    for ballot in ballots:
        for group in ballot["ranking"]:
            candidates.update(group)

    candidates = sorted(candidates)
    pairwise_wins = {candidate: 0 for candidate in candidates}

    for i in range(len(candidates)):
        for j in range(i + 1, len(candidates)):
            candidate_i = candidates[i]
            candidate_j = candidates[j]
            votes_for_i = 0
            votes_for_j = 0

            for ballot in ballots:
                count = ballot["count"]
                ranking = ballot["ranking"]
                position_i = next((idx for idx, group in enumerate(ranking) if candidate_i in group), float('inf'))
                position_j = next((idx for idx, group in enumerate(ranking) if candidate_j in group), float('inf'))

                if position_i < position_j:
                    votes_for_i += count
                elif position_j < position_i:
                    votes_for_j += count

            if votes_for_i > votes_for_j:
                pairwise_wins[candidate_i] += 1
            elif votes_for_j > votes_for_i:
                pairwise_wins[candidate_j] += 1

    num_candidates = len(candidates)
    winners = [candidate for candidate, wins in pairwise_wins.items() if wins == num_candidates - 1]
    return winners

In [15]:
condorcet_winner_idx = condorcet_winner_SCF(ballots)
for idx in condorcet_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

Jean-Luc Mélenchon
